        # GraphRAG on PDF Documents

        This notebook is designed for a hands-on workshop where participants have already seen standard RAG and now need a practical introduction to **GraphRAG**.

        We will use the **official Microsoft GraphRAG workflow** and add one realistic preprocessing step: converting PDF files into plain text, because GraphRAG officially supports **text, CSV, and JSON** inputs rather than raw PDFs.

        ## Scenario

        You have one or more PDF files and want to answer questions that require more than simple chunk retrieval. In particular, you want to:

        - connect entities and relationships across distant parts of the corpus
        - answer local questions about specific entities
        - answer broader questions about themes or communities in the corpus
        - inspect what the system extracted, rather than treating the pipeline like a black box
        

        ## Learning Objectives

        By the end of this notebook, participants should be able to:

        1. explain when GraphRAG is useful compared with standard RAG
        2. prepare a PDF corpus for GraphRAG ingestion
        3. build a GraphRAG project using the official open-source package
        4. inspect the generated graph artifacts such as entities, relationships, and community reports
        5. run local, global, and basic queries against the indexed corpus
        6. discuss the main practical tradeoffs of GraphRAG in real projects
        

        ## Before We Start

        **Audience assumption**

        This notebook assumes participants already understand:

        - embeddings
        - chunking
        - vector stores
        - the standard RAG flow of `ingest -> embed -> retrieve -> generate`

        **What we will not go deep into**

        We will not fully unpack:

        - graph theory in depth
        - prompt tuning internals for every GraphRAG stage
        - custom vector stores or custom graph backends
        - production deployment architecture

        That keeps the session practical without becoming too lightweight.
        

        ## Why GraphRAG After Standard RAG?

        Standard RAG is strong when the answer is already sitting in a few nearby chunks.

        GraphRAG becomes attractive when:

        - the answer requires connecting facts spread across many parts of the corpus
        - the question is about relationships between people, teams, systems, events, or concepts
        - you want a higher-level understanding of themes in a collection of documents
        - you want both graph-aware retrieval and source-grounded text evidence

        A useful teaching shorthand is:

        - **Standard RAG**: "retrieve the right chunks"
        - **GraphRAG**: "retrieve the right chunks plus the relationships and community structure around them"
        

        ## Chosen Implementation

        For this workshop we will use **Microsoft GraphRAG**, because it is the most recognizable open-source implementation that directly centers the GraphRAG pattern.

        We will follow the official workflow:

        1. prepare input documents
        2. initialize a GraphRAG project
        3. run indexing
        4. inspect graph outputs
        5. query with local/global/basic search

        **Important practical note**

        The official GraphRAG documentation supports these input types out of the box:

        - plain text files
        - CSV
        - JSON

        So for PDF documents, we first extract the PDF text and save each document as a `.txt` file.
        

        ## References Used for This Notebook

        These are the main official sources behind the workflow used here:

        - Microsoft GraphRAG docs: https://microsoft.github.io/graphrag/
        - GraphRAG input formats: https://microsoft.github.io/graphrag/index/inputs/
        - GraphRAG CLI reference: https://microsoft.github.io/graphrag/cli/
        - GraphRAG local search: https://microsoft.github.io/graphrag/query/local_search/

        Those references are helpful if participants want to extend this notebook after class.
        

In [ ]:
        # If you are running this notebook in a fresh environment, install the essentials.
        # Uncomment and run if needed.

        # %pip install -U graphrag pypdf pandas pyarrow matplotlib networkx python-dotenv
        

        ## Environment Setup

        GraphRAG needs LLM access for extraction, summarization, and query-time reasoning.

        In the simplest classroom setup, store your key in an environment variable named:

        - `GRAPHRAG_API_KEY`

        Depending on your provider setup, you may also need to edit the generated `settings.yaml` after initialization.
        

In [ ]:
        import os
        from pathlib import Path

        PROJECT_DIR = Path("graphrag_pdf_demo")
        PDF_DIR = Path("data/pdfs")          # Put your PDF files here
        TEXT_INPUT_DIR = PROJECT_DIR / "input"
        OUTPUT_DIR = PROJECT_DIR / "output"

        PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        PDF_DIR.mkdir(parents=True, exist_ok=True)
        TEXT_INPUT_DIR.mkdir(parents=True, exist_ok=True)

        print("Project directory :", PROJECT_DIR.resolve())
        print("PDF directory     :", PDF_DIR.resolve())
        print("Text input dir    :", TEXT_INPUT_DIR.resolve())

        if not os.getenv("GRAPHRAG_API_KEY"):
            print("\nWARNING: GRAPHRAG_API_KEY is not set yet.")
            print("Set it before running indexing or query steps.")
        

        ## Step 1: Convert PDF Files into GraphRAG-Friendly Text Files

        This is the only extra step we need because GraphRAG does not directly ingest PDF files out of the box.

        The function below:

        - reads every PDF in `data/pdfs/`
        - extracts text page by page
        - writes one `.txt` file per PDF into the GraphRAG input folder
        - adds lightweight source metadata at the top of each text file

        That small metadata header often helps later when you inspect retrieved text units.
        

In [ ]:
        from pypdf import PdfReader


        def pdf_to_text_files(pdf_dir: Path, text_dir: Path) -> list[Path]:
            created_files = []

            for pdf_path in sorted(pdf_dir.glob("*.pdf")):
                reader = PdfReader(str(pdf_path))
                page_text = []

                for page_number, page in enumerate(reader.pages, start=1):
                    extracted = page.extract_text() or ""
                    cleaned = extracted.strip()
                    if cleaned:
                        page_text.append(f"\n\n[Page {page_number}]\n{cleaned}")

                final_text = "\n".join(
                    [
                        f"source_file: {pdf_path.name}",
                        f"page_count: {len(reader.pages)}",
                        "",
                        "\n".join(page_text).strip(),
                    ]
                ).strip()

                output_path = text_dir / f"{pdf_path.stem}.txt"
                output_path.write_text(final_text, encoding="utf-8")
                created_files.append(output_path)

            return created_files


        created_text_files = pdf_to_text_files(PDF_DIR, TEXT_INPUT_DIR)
        print("Created text files:")
        for path in created_text_files:
            print("-", path)

        if not created_text_files:
            print("No PDFs found yet. Add one or more PDF files to data/pdfs/ and rerun this cell.")
        

In [ ]:
        # Preview one extracted file so participants can see what GraphRAG will actually ingest.

        if created_text_files:
            sample_text = created_text_files[0].read_text(encoding="utf-8")
            print(sample_text[:3000])
        

        ## Step 2: Initialize a GraphRAG Project

        The official package provides an `init` command that creates the default project structure and configuration.

        We will point it at `PROJECT_DIR`. If you rerun this notebook often, keep `--force` commented out unless you really want to overwrite the current setup.
        

In [ ]:
        import subprocess


        def run(cmd: list[str], cwd: Path | None = None) -> subprocess.CompletedProcess:
            print("$", " ".join(cmd))
            result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
            print(result.stdout)
            if result.returncode != 0:
                print(result.stderr)
            result.check_returncode()
            return result


        # First-time setup:
        # run(["graphrag", "init", "--root", str(PROJECT_DIR)])

        # If you need to reinitialize from scratch:
        # run(["graphrag", "init", "--root", str(PROJECT_DIR), "--force"])
        

        ## Step 3: Review the Generated Configuration

        After `graphrag init`, inspect `settings.yaml`.

        For teaching purposes, focus on these sections:

        - model definitions
        - input settings
        - chunk settings
        - local search settings
        - global search settings

        In many classroom runs, the defaults are enough to start. The most common edits are:

        - provider and model names
        - API base or deployment details
        - chunk size or overlap
        - token limits for local/global search
        

In [ ]:
        settings_path = PROJECT_DIR / "settings.yaml"
        if settings_path.exists():
            print(settings_path.read_text(encoding="utf-8")[:5000])
        else:
            print("Run the init step first so settings.yaml is created.")
        

        ## Recommended Teaching Notes on Configuration

        Keep the explanation at this level in the live session:

        - **Input** tells GraphRAG where the source text lives.
        - **Chunking** decides how documents become text units.
        - **Extraction** pulls out entities and relationships.
        - **Community detection** groups related parts of the graph.
        - **Community reports** create higher-level summaries.
        - **Local search** is great for specific entity-centric questions.
        - **Global search** is better for broad, corpus-level questions.

        That is enough detail for a first GraphRAG workshop without overwhelming people.
        

        ## Step 4: Run Indexing

        This is the stage where GraphRAG does the heavy lifting:

        - reads the prepared text files
        - creates text units
        - extracts entities and relationships
        - builds communities
        - generates community reports
        - stores intermediate and final artifacts

        The official CLI supports multiple indexing methods. For a workshop, start with the default standard method.
        

In [ ]:
        # Dry run is a good classroom move because it validates the configuration before spending tokens.
        # run(["graphrag", "index", "--root", str(PROJECT_DIR), "--dry-run"])

        # Actual indexing:
        # run(["graphrag", "index", "--root", str(PROJECT_DIR), "--method", "standard"])
        

        ## Step 5: Inspect the Output Artifacts

        One of the nice teaching advantages of GraphRAG is that the pipeline produces inspectable data artifacts.

        After indexing, the output folder usually contains parquet tables that let you answer:

        - what documents were loaded?
        - what text units were created?
        - which entities were extracted?
        - which relationships were found?
        - what communities were detected?
        - what summaries were written about those communities?
        

In [ ]:
        import pandas as pd

        if OUTPUT_DIR.exists():
            parquet_files = sorted(OUTPUT_DIR.rglob("*.parquet"))
            print("Parquet outputs:")
            for file in parquet_files:
                print("-", file.relative_to(PROJECT_DIR))
        else:
            print("No output folder found yet. Run indexing first.")
        

In [ ]:
        def load_first_matching_parquet(pattern: str):
            matches = sorted(OUTPUT_DIR.rglob(pattern))
            if not matches:
                print(f"No parquet file found for pattern: {pattern}")
                return None
            path = matches[0]
            print("Loading:", path)
            return pd.read_parquet(path)


        documents_df = load_first_matching_parquet("*documents*.parquet")
        text_units_df = load_first_matching_parquet("*text_units*.parquet")
        entities_df = load_first_matching_parquet("*entities*.parquet")
        relationships_df = load_first_matching_parquet("*relationships*.parquet")
        communities_df = load_first_matching_parquet("*communities*.parquet")
        reports_df = load_first_matching_parquet("*community_reports*.parquet")
        

In [ ]:
        if documents_df is not None:
            display(documents_df.head())

        if text_units_df is not None:
            display(text_units_df.head())

        if entities_df is not None:
            display(entities_df.head())

        if relationships_df is not None:
            display(relationships_df.head())

        if communities_df is not None:
            display(communities_df.head())

        if reports_df is not None:
            display(reports_df.head())
        

        ## Interpreting the Artifacts

        Here is a simple way to explain the outputs to participants:

        - `documents`: the source documents that entered the pipeline
        - `text_units`: the chunked units GraphRAG uses internally
        - `entities`: the important nouns or concepts extracted from text units
        - `relationships`: how those entities are connected
        - `communities`: clusters of related entities
        - `community_reports`: LLM-written summaries of those clusters

        This is the main conceptual leap from standard RAG: retrieval is no longer driven only by chunk similarity.
        

In [ ]:
        # A quick graph view for teaching.
        # This is not required by GraphRAG, but it helps participants "see" the extracted structure.

        import matplotlib.pyplot as plt
        import networkx as nx

        if relationships_df is not None and {"source", "target"}.issubset(relationships_df.columns):
            sample_edges = relationships_df.head(40)
            G = nx.from_pandas_edgelist(sample_edges, source="source", target="target")

            plt.figure(figsize=(10, 8))
            pos = nx.spring_layout(G, seed=42)
            nx.draw_networkx(G, pos=pos, with_labels=True, node_size=1500, font_size=8)
            plt.title("Sample Relationship Graph")
            plt.axis("off")
            plt.show()
        else:
            print("Relationship data not available yet.")
        

        ## Step 6: Query the GraphRAG Index

        GraphRAG supports multiple search styles. For teaching, these three are enough:

        - **basic**: closest to standard vector-style retrieval
        - **local**: entity-centric reasoning with graph context
        - **global**: broad reasoning over community reports

        A nice classroom pattern is to ask the same topic in multiple ways and compare the behavior.
        

In [ ]:
        def query_graphrag(question: str, method: str = "local"):
            return run(
                [
                    "graphrag",
                    "query",
                    "--root",
                    str(PROJECT_DIR),
                    "--method",
                    method,
                    question,
                ]
            )


        # Examples:
        # query_graphrag("Who are the main people discussed in the documents?", method="local")
        # query_graphrag("What are the main themes across the whole PDF collection?", method="global")
        # query_graphrag("What does the document say about policy changes?", method="basic")
        

        ## Suggested Demo Questions

        Once your PDFs are indexed, try questions like these:

        **Local search questions**

        - Who is `<person/entity>` and how are they connected to the rest of the document set?
        - What does the corpus say about `<specific concept>`?
        - Which events are associated with `<organization or topic>`?

        **Global search questions**

        - What are the major themes across all uploaded PDFs?
        - How do the documents collectively describe the main challenges or opportunities?
        - What high-level story emerges from the corpus?

        **Basic search questions**

        - Where is `<fact>` mentioned?
        - What does the source text say directly about `<topic>`?
        

        ## A Practical Comparison to Standard RAG

        Use this simple comparison in your teaching:

        **Standard RAG**

        - fast to explain
        - easier to build
        - strong for direct lookup questions
        - weaker when the answer needs multi-hop connections

        **GraphRAG**

        - more moving parts
        - higher indexing cost
        - more interpretable structure
        - stronger for entity networks, theme discovery, and synthesis over larger corpora
        

        ## Common Issues to Mention in Class

        1. **Poor PDF extraction**
           If the PDF is image-heavy or scanned, text extraction may be noisy. OCR may be required first.

        2. **Messy entities**
           Entity extraction quality depends heavily on document clarity and model quality.

        3. **Token and cost overhead**
           GraphRAG indexing is much more expensive than plain chunk embedding.

        4. **Configuration confusion**
           Participants can get lost in settings. Keep the first session focused on only the most important knobs.

        5. **Not every question needs GraphRAG**
           Use it where relationship-aware retrieval genuinely helps.
        

        ## Exercises

        1. Add two or three PDF files from the same topic area and compare the entity graph.
        2. Ask one local question and one global question about the same corpus. Explain why the answers differ.
        3. Modify chunk settings and rerun indexing. Inspect whether the extracted entities or relationships improve.
        4. Compare one answer from GraphRAG with the answer from your earlier standard RAG pipeline.
        

        ## Summary

        In this notebook we used a practical GraphRAG pipeline:

        1. convert PDFs to text
        2. initialize the official Microsoft GraphRAG project
        3. run indexing
        4. inspect graph artifacts
        5. query with local, global, and basic search

        That gives participants a realistic mental model without going too far into advanced internals.

        ## Next Steps

        Good follow-up topics after this workshop:

        - prompt tuning for GraphRAG
        - comparing GraphRAG against a baseline RAG benchmark set
        - using a custom vector store or graph database backend
        - building a small app interface on top of the indexed corpus
        